# Day 17 - Composite Score & Company Ranking

## Objectives

- Load financial ratio data
- Select latest financial record for each company
- Normalize financial metrics to a 0-100 scale
- Apply metric weights
- Calculate composite financial score
- Generate company rankings
- Classify companies into score bands
- Export scoring and ranking reports

In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
df = pd.read_csv(
    "../reports/financial_ratios_day12.csv"
)

print("Financial Ratio Data Loaded Successfully")
print("Shape:", df.shape)

df.head()

Financial Ratio Data Loaded Successfully
Shape: (1184, 7)


,company_id,year,return_on_equity_pct,debt_to_equity,interest_coverage,asset_turnover,dividend_payout_ratio
0,ABB,Dec 2012,22.411128,0.0,NaN,1.822492,17.241379
1,ABB,Mar 2014,25.126904,0.0,NaN,1.998244,12.626263
2,ABB,Mar 2014,25.126904,0.0,NaN,1.998244,12.626263
3,ABB,Mar 2015,24.439701,0.0,NaN,1.665939,12.663755
4,ABB,Mar 2015,24.439701,0.0,NaN,1.665939,12.663755


In [3]:
numeric_cols = [
    "return_on_equity_pct",
    "debt_to_equity",
    "interest_coverage",
    "asset_turnover",
    "dividend_payout_ratio"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

print(df[numeric_cols].dtypes)

return_on_equity_pct     float64
debt_to_equity           float64
interest_coverage        float64
asset_turnover           float64
dividend_payout_ratio    float64
dtype: object


In [4]:
df["year_number"] = pd.to_numeric(
    df["year"]
    .astype(str)
    .str.extract(r"(\d{4})")[0],
    errors="coerce"
)

latest_df = (
    df.sort_values("year_number")
      .drop_duplicates(
          subset=["company_id"],
          keep="last"
      )
      .reset_index(drop=True)
)

print("Original Records:", len(df))
print("Latest Company Records:", len(latest_df))

latest_df[[
    "company_id",
    "year",
    "year_number"
]].head()

Original Records: 1184
Latest Company Records: 92


,company_id,year,year_number
0,HCLTECH,Mar 2024,2024
1,BHEL,Mar 2024,2024
2,HAVELLS,Mar 2024,2024
3,PFC,Mar 2024,2024
4,HDFCLIFE,Mar 2024,2024


In [5]:
latest_df[numeric_cols].isnull().sum()

return_on_equity_pct     0
debt_to_equity           0
interest_coverage        1
asset_turnover           0
dividend_payout_ratio    3
dtype: int64

In [6]:
for col in numeric_cols:
    latest_df[col] = latest_df[col].fillna(
        latest_df[col].median()
    )

print(
    latest_df[numeric_cols]
    .isnull()
    .sum()
)

return_on_equity_pct     0
debt_to_equity           0
interest_coverage        0
asset_turnover           0
dividend_payout_ratio    0
dtype: int64


In [7]:
def normalize_positive(series):
    min_value = series.min()
    max_value = series.max()

    if max_value == min_value:
        return pd.Series(
            50,
            index=series.index
        )

    return (
        (series - min_value) /
        (max_value - min_value)
    ) * 100

In [8]:
def normalize_negative(series):
    min_value = series.min()
    max_value = series.max()

    if max_value == min_value:
        return pd.Series(
            50,
            index=series.index
        )

    return (
        (max_value - series) /
        (max_value - min_value)
    ) * 100

In [9]:
latest_df["roe_score"] = normalize_positive(
    latest_df["return_on_equity_pct"]
)

latest_df["debt_score"] = normalize_negative(
    latest_df["debt_to_equity"]
)

latest_df["interest_score"] = normalize_positive(
    latest_df["interest_coverage"]
)

latest_df["asset_turnover_score"] = normalize_positive(
    latest_df["asset_turnover"]
)

latest_df["dividend_score"] = normalize_positive(
    latest_df["dividend_payout_ratio"]
)

In [10]:
latest_df[[
    "company_id",
    "roe_score",
    "debt_score",
    "interest_score",
    "asset_turnover_score",
    "dividend_score"
]].head()

,company_id,roe_score,debt_score,interest_score,asset_turnover_score,dividend_score
0,HCLTECH,0.596896,99.432650,1.628167,0.871783,5.211392
1,BHEL,0.136625,97.562546,0.029304,0.307307,100.000000
2,HAVELLS,0.471687,99.726331,0.831891,1.177680,31.491587
3,PFC,0.663157,42.680948,0.018888,0.059804,0.790695
4,HDFCLIFE,0.338302,99.564311,7.163937,0.256178,15.604378


In [11]:
score_cols = [
    "roe_score",
    "debt_score",
    "interest_score",
    "asset_turnover_score",
    "dividend_score"
]

for col in score_cols:
    print(
        col,
        "Min:",
        latest_df[col].min(),
        "Max:",
        latest_df[col].max()
    )

roe_score Min: 0.0 Max: 100.0
debt_score Min: 0.0 Max: 100.0
interest_score Min: 0.0 Max: 100.0
asset_turnover_score Min: 0.0 Max: 100.0
dividend_score Min: 0.0 Max: 100.0


In [12]:
weights = {
    "roe_score": 0.30,
    "debt_score": 0.25,
    "interest_score": 0.20,
    "asset_turnover_score": 0.15,
    "dividend_score": 0.10
}

print("Total Weight:", sum(weights.values()))

Total Weight: 1.0


In [13]:
latest_df["composite_score"] = (
    latest_df["roe_score"] * weights["roe_score"]
    +
    latest_df["debt_score"] * weights["debt_score"]
    +
    latest_df["interest_score"] * weights["interest_score"]
    +
    latest_df["asset_turnover_score"] * weights["asset_turnover_score"]
    +
    latest_df["dividend_score"] * weights["dividend_score"]
)

latest_df[[
    "company_id",
    "year",
    "composite_score"
]].head()

,company_id,year,composite_score
0,HCLTECH,Mar 2024,26.014771
1,BHEL,Mar 2024,34.483581
2,HAVELLS,Mar 2024,28.565278
3,PFC,Mar 2024,10.961002
4,HDFCLIFE,Mar 2024,28.024220


In [14]:
latest_df["company_rank"] = (
    latest_df["composite_score"]
    .rank(
        method="dense",
        ascending=False
    )
    .astype(int)
)

latest_df = latest_df.sort_values(
    by="company_rank"
).reset_index(drop=True)

latest_df[[
    "company_id",
    "year",
    "composite_score",
    "company_rank"
]].head(20)

,company_id,year,composite_score,company_rank
0,BEL,Mar 2024,73.189325,1
1,HAL,Mar 2024,58.206714,2
2,BAJFINANCE,Mar 2024,38.831554,3
3,INDIGO,Mar 2024,37.526171,4
4,BHEL,Mar 2024,34.483581,5
5,NAUKRI,Mar 2024,32.681642,6
6,DIVISLAB,Mar 2024,32.127323,7
7,ABB,Mar 2024,31.765038,8
8,BAJAJHLDNG,Mar 2024,31.187216,9
9,TECHM,Mar 2024,30.850326,10


In [15]:
def classify_score(score):
    if score >= 80:
        return "Excellent"
    elif score >= 60:
        return "Strong"
    elif score >= 40:
        return "Average"
    elif score >= 20:
        return "Weak"
    else:
        return "Poor"

In [16]:
latest_df["score_band"] = (
    latest_df["composite_score"]
    .apply(classify_score)
)

latest_df[[
    "company_id",
    "composite_score",
    "company_rank",
    "score_band"
]].head(20)

,company_id,composite_score,company_rank,score_band
0,BEL,73.189325,1,Strong
1,HAL,58.206714,2,Average
2,BAJFINANCE,38.831554,3,Weak
3,INDIGO,37.526171,4,Weak
4,BHEL,34.483581,5,Weak
5,NAUKRI,32.681642,6,Weak
6,DIVISLAB,32.127323,7,Weak
7,ABB,31.765038,8,Weak
8,BAJAJHLDNG,31.187216,9,Weak
9,TECHM,30.850326,10,Weak


In [17]:
score_band_summary = (
    latest_df["score_band"]
    .value_counts()
    .reset_index()
)

score_band_summary.columns = [
    "score_band",
    "company_count"
]

score_band_summary

,score_band,company_count
0,Weak,74
1,Poor,16
2,Strong,1
3,Average,1


In [18]:
top20_companies = latest_df[
    [
        "company_id",
        "year",
        "return_on_equity_pct",
        "debt_to_equity",
        "interest_coverage",
        "asset_turnover",
        "dividend_payout_ratio",
        "roe_score",
        "debt_score",
        "interest_score",
        "asset_turnover_score",
        "dividend_score",
        "composite_score",
        "company_rank",
        "score_band"
    ]
].head(20)

top20_companies

,company_id,year,return_on_equity_pct,debt_to_equity,interest_coverage,asset_turnover,dividend_payout_ratio,roe_score,debt_score,interest_score,asset_turnover_score,dividend_score,composite_score,company_rank,score_band
0,BEL,Mar 2024,4744.047619,0.511905,420.916667,125.888199,1.003764,100.000000,96.556869,15.685033,100.000000,9.131016,73.189325,1,Strong
1,HAL,Mar 2024,3816.582915,0.618090,226.720930,67.513333,0.408163,80.471889,95.842652,8.447280,53.624880,3.712969,58.206714,2,Average
2,BAJFINANCE,Mar 2024,18.841921,3.824789,2683.166667,0.146303,0.103799,0.509052,74.274025,100.000000,0.106055,0.944236,38.831554,3,Weak
3,INDIGO,Mar 2024,892.568306,0.018579,22.602322,56.386252,0.000000,18.905684,99.875034,0.839698,44.785121,0.000000,37.526171,4,Weak
4,BHEL,Mar 2024,1.153941,0.362386,0.858696,0.399629,10.992908,0.136625,97.562546,0.029304,0.307307,100.000000,34.483581,5,Weak
5,NAUKRI,Mar 2024,1.966162,0.008129,20.314286,0.070271,8.235294,0.153727,99.945323,0.754422,0.045653,74.914611,32.681642,6,Weak
6,DIVISLAB,Mar 2024,11.789846,0.000221,552.500000,0.507439,3.125000,0.360568,99.998513,20.589197,0.392955,28.427419,32.127323,7,Weak
7,ABB,Mar 2024,32.468235,0.022438,121.083333,1.126324,6.078268,0.795960,99.849076,4.510125,0.884620,55.292632,31.765038,8,Weak
8,BAJAJHLDNG,Mar 2024,13.576788,0.001161,781.000000,0.026120,0.271555,0.398193,99.992189,29.105484,0.010578,2.470271,31.187216,9,Weak
9,TECHM,Mar 2024,8.987964,0.095129,11.494898,1.205034,6.257822,0.301574,99.360150,0.425720,0.947150,56.925996,30.850326,10,Weak


In [19]:
latest_df.to_csv(
    "../reports/day17_composite_scores.csv",
    index=False
)

top20_companies.to_csv(
    "../reports/day17_top20_companies.csv",
    index=False
)

score_band_summary.to_csv(
    "../reports/day17_score_band_summary.csv",
    index=False
)

print("Day 17 Composite Score Reports Saved Successfully!")

Day 17 Composite Score Reports Saved Successfully!


# Day 17 Summary

## Composite Scoring Engine Completed

- Loaded financial ratio data.
- Selected the latest available record for each company.
- Handled missing financial metrics.
- Normalized financial ratios to a 0-100 scale.
- Applied reverse scoring for Debt-to-Equity.
- Defined financial metric weights.
- Calculated weighted composite scores.
- Generated company rankings.
- Classified companies into score bands.
- Generated Top 20 company report.
- Created a reusable composite scoring module.

## Sprint 3 Progress

The screener and preset framework now includes a composite financial scoring and company ranking engine.